# 🎓 SmartEvent: Machine Learning Attendance Prediction Model
**Author:** Elvis Wanyama Wafuke (154210)  
**Institution:** Strathmore University - School of Computing and Engineering Sciences  
**Project:** Capstone Project — SmartEvent  

---  
### 📌 Overview
This Google Colab notebook implements the machine learning attendance prediction pipeline for university events:
1. **Google Drive Integration**: Mounts Google Drive to store datasets and serialized `.joblib` model artifacts.
2. **Data Generation & Preprocessing**: Synthesizes realistic Strathmore event data (academic, workshops, seminars, sports, career fairs) with features like venue capacity, registration count, time slot, past attendance rate, and priority SMS status.
3. **Model Training & Evaluation**: Trains Logistic Regression and Random Forest Regressor models, evaluating MAE, R² score, and prediction accuracy.
4. **Export to Google Drive**: Serializes the model pipeline using `joblib` directly to `/content/drive/MyDrive/SmartEvent_ML/models/attendance_model.joblib`.

## 📁 Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define Google Drive project directory paths
DRIVE_BASE_DIR = '/content/drive/MyDrive/SmartEvent_ML'
DATASET_DIR = os.path.join(DRIVE_BASE_DIR, 'datasets')
MODEL_DIR = os.path.join(DRIVE_BASE_DIR, 'models')

# Create directories in Google Drive if they don't exist
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"✅ Google Drive Mounted Successfully!")
print(f"📂 Datasets Path: {DATASET_DIR}")
print(f"📂 Models Path:   {MODEL_DIR}")

## 📊 Step 2: Generate & Load Strathmore Event Dataset

In [ ]:
import numpy as np
import pandas as pd

def generate_strathmore_event_dataset(num_samples=1500, seed=42):
    np.random.seed(seed)
    
    event_types = ['Academic', 'Workshop', 'Seminar', 'Club Activity', 'Sports', 'Career Fair']
    time_slots = ['Morning', 'Afternoon', 'Evening']
    
    records = []
    for _ in range(num_samples):
        e_type = np.random.choice(event_types, p=[0.25, 0.25, 0.20, 0.15, 0.10, 0.05])
        day_of_week = np.random.randint(0, 7) # 0=Mon, 6=Sun
        time_slot = np.random.choice(time_slots, p=[0.3, 0.5, 0.2])
        venue_capacity = int(np.random.choice([50, 80, 150, 200, 350, 500]))
        
        registration_count = int(venue_capacity * np.random.uniform(0.5, 1.15))
        past_attendance_rate = np.round(np.random.uniform(0.45, 0.90), 2)
        sms_priority_sent = int(np.random.choice([1, 2, 3], p=[0.3, 0.4, 0.3])) # 1: Low, 2: Medium, 3: High
        
        # Base turnout calculation logic
        base_rate = past_attendance_rate
        if e_type in ['Academic', 'Career Fair']:
            base_rate += 0.08
        elif e_type == 'Sports':
            base_rate += 0.05
        elif e_type == 'Seminar':
            base_rate -= 0.04
            
        # Priority SMS Impact
        if sms_priority_sent == 3:
            base_rate += 0.12 # High Priority SMS boosts turnout
        elif sms_priority_sent == 2:
            base_rate += 0.05
            
        if day_of_week in [1, 2, 3]: # Tue-Thu peak
            base_rate += 0.04
        elif day_of_week in [5, 6]: # Weekends drop
            base_rate -= 0.10
            
        turnout_rate = np.clip(base_rate + np.random.normal(0, 0.04), 0.35, 0.98)
        actual_attendance = int(np.clip(registration_count * turnout_rate, 10, venue_capacity))
        
        records.append({
            'eventType': e_type,
            'dayOfWeek': day_of_week,
            'timeSlot': time_slot,
            'venueCapacity': venue_capacity,
            'registrationCount': registration_count,
            'pastAttendanceRate': past_attendance_rate,
            'smsPriority': sms_priority_sent,
            'attendanceCount': actual_attendance,
            'turnoutRate': round(turnout_rate, 3)
        })
        
    return pd.DataFrame(records)

# Generate & Save to Google Drive
df = generate_strathmore_event_dataset(1500)
csv_drive_path = os.path.join(DATASET_DIR, 'strathmore_events_dataset.csv')
df.to_csv(csv_drive_path, index=False)
print(f"✅ Generated {len(df)} Strathmore event records and saved to Google Drive:")
print(f"📄 {csv_drive_path}")
df.head()

## 🤖 Step 3: Feature Engineering & Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

X = df[['eventType', 'dayOfWeek', 'timeSlot', 'venueCapacity', 'registrationCount', 'pastAttendanceRate', 'smsPriority']]
y = df['attendanceCount']

categorical_features = ['eventType', 'timeSlot']
numerical_features = ['dayOfWeek', 'venueCapacity', 'registrationCount', 'pastAttendanceRate', 'smsPriority']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ]
)

# Define Random Forest Regressor Pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42))
])

# Split into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
model_pipeline.fit(X_train, y_train)

# Predict on Evaluation Set
predictions = model_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("===========================================")
print("🎯 MODEL EVALUATION METRICS")
print(f"Mean Absolute Error (MAE): {mae:.2f} attendees")
print(f"R² Score (Variance Explained): {r2:.4f} ({round(r2*100, 2)}% accuracy)")
print("===========================================")

## 💾 Step 4: Export Serialized Model to Google Drive

In [ ]:
import joblib

model_drive_filename = os.path.join(MODEL_DIR, 'attendance_model.joblib')
joblib.dump(model_pipeline, model_drive_filename)

print(f"🎉 SUCCESS! Model saved directly to your Google Drive:")
print(f"💾 Location: {model_drive_filename}")
print(f"Copy this .joblib file to your local smartevent/ml-service/models/ directory or point ML_SERVICE_URL to it.")